# 09 — Step 7: collaborative filtering (CF)

**Purpose.** vibe_match learns only from a user's own taste in vibe. CF learns from patterns across all users: people who loved *Amélie* also tend to love *Lost in Translation*, even when the vibes differ. The questions:
1. Which CF method works best on our benchmark?
2. Does **CF + vibe** beat either alone? That is the core claim of VibeRec.

**Methods** (Adil's choices, 2026-10-09)

| Method | Idea |
|---|---|
| **Item-to-item similarity** | Movies rated by the same people are similar; recommend movies similar to the user's. The simplest CF, so the others must beat it. |
| **PureSVD** | Compress the users × movies table into k hidden factors; score unseen movies by how well they fit the user's rated ones. Replaces `Surprise`'s SVD, which targets star accuracy and is too slow at ranking. |
| **ALS** (`implicit`) | Learns hidden factors for every user and movie from who-watched-what, with a confidence weight (alpha). Built for top-N. |
| **CF + vibe blend** | Weighted mix of the best CF scores and vibe_match scores. Answers question 2; LightFM, which would take vibe features directly, does not build on this Python. |

**Each method is tried on three versions of the rating table:**
- **watched:** every rating counts;
- **liked:** only ratings above the user's own mean;
- **rating:** the star values (PureSVD only).

**No peeking.** Settings (factors, regularisation, alpha, neighbours, blend weight) are tuned on a **validation split** (cutoff 2016-06-25, predicting mid-2016 to the start of 2018 from pre-2016 data). Selection is by recall@10, ties broken by recall@50, decided before seeing results. Only the tuned version of each method is then scored once in the **testing zone** (2018 cutoff, the same 5,000 users as notebooks 07 and 08) and added to the scoreboard. Everything is trained on ratings before the relevant cutoff only.

## 1. Config and data

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"          # implicit's advice: its own threads, not BLAS threads
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

sys.path.insert(0, str(Path("..").resolve()))
from src.viberec import cf, evaluation as ev, recommenders as rc

PROCESSED_DIR = Path("..") / "data" / "processed"
ARTIFACTS_DIR = Path("..") / "artifacts"
MODELS_DIR = Path("..") / "models"
RATINGS_CSV = Path("..") / "data" / "raw" / "movielens" / "ml-25m" / "ratings.csv"
SCOREBOARD = ARTIFACTS_DIR / "scoreboard.csv"
VERSION, SEED, K_SHRINK, N_VAL_USERS = "v1", 42, 7.324, 3000


def write_metadata(path, **info):
    meta = {"created_at": datetime.now(timezone.utc).isoformat(), "produced_by": "notebooks/09_collaborative_filtering.ipynb", **info}
    path.with_name(path.stem + ".metadata.json").write_text(json.dumps(meta, indent=2, default=str))


early_all = pd.read_parquet(ARTIFACTS_DIR / f"q43_early_counts_{VERSION}.parquet")
cut = {q: int(early_all.loc[early_all["quantile"] == q, "cutoff_ts"].iloc[0]) for q in (0.8, 0.9)}
VAL_CUTOFF, TEST_CUTOFF = cut[0.8], cut[0.9]
print("validation cutoff", pd.to_datetime(VAL_CUTOFF, unit="s").date(), "| test cutoff", pd.to_datetime(TEST_CUTOFF, unit="s").date())

PRE_PATH = PROCESSED_DIR / f"ratings_before_2018_{VERSION}.parquet"
if PRE_PATH.exists():
    print(f"{PRE_PATH} already exists, loading.")
    pre2018 = pd.read_parquet(PRE_PATH)
else:
    ratings = pd.read_csv(RATINGS_CSV, dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"})
    pre2018 = ratings[ratings["timestamp"] < TEST_CUTOFF].reset_index(drop=True)
    pre2018.to_parquet(PRE_PATH, index=False)
    write_metadata(PRE_PATH, cutoff=str(pd.to_datetime(TEST_CUTOFF, unit="s")), rows=len(pre2018),
                   note="all MovieLens ratings before the testing-zone cutoff; CF training data for the test")
    del ratings
pre2016 = pre2018[pre2018["timestamp"] < VAL_CUTOFF]
print(f"ratings before 2018: {len(pre2018):,}; before mid-2016: {len(pre2016):,}")

### Validation split (for tuning) and the testing zone (for the final score)

- **Validation:** 3,000 users with ratings before mid-2016 and at least one hit between mid-2016 and the 2018 cutoff (the test period is never touched). Pool = tier 3 at the 2016 cutoff.
- **Test:** the saved testing-zone benchmark (5,000 users, 2018 cutoff), pool = tier 3 at the 2018 cutoff, the same as `popular / tier 3` on the scoreboard.

In [ ]:
movies = pd.read_parquet(PROCESSED_DIR / "movies_full.parquet")
movies["genres"] = movies["genres"].str.replace(",", "|", regex=False)

VAL_TRAIN, VAL_HITS = PROCESSED_DIR / f"validation_train_{VERSION}.parquet", PROCESSED_DIR / f"validation_hits_{VERSION}.parquet"
if VAL_TRAIN.exists() and VAL_HITS.exists():
    print("validation split already exists, loading.")
    val_train, val_hits_df = pd.read_parquet(VAL_TRAIN), pd.read_parquet(VAL_HITS)
else:
    val_train, vh, _, eligible = ev.make_benchmark(pre2018, VAL_CUTOFF, n_users=N_VAL_USERS, seed=SEED)
    val_hits_df = pd.DataFrame([(u, m) for u, ms in vh.items() for m in ms], columns=["userId", "movieId"])
    val_train.to_parquet(VAL_TRAIN, index=False)
    val_hits_df.to_parquet(VAL_HITS, index=False)
    for p in (VAL_TRAIN, VAL_HITS):
        write_metadata(p, cutoff=str(pd.to_datetime(VAL_CUTOFF, unit="s")), users=N_VAL_USERS, seed=SEED,
                       eligible_users=eligible, note="tuning split: pre-2016 ratings, hits between mid-2016 and 2018-01-04")
val_hits = val_hits_df.groupby("userId")["movieId"].apply(set).to_dict()
val_mean = val_train.groupby("userId")["rating"].mean()
val_known = ev.movies_known_at(early_all[early_all["quantile"] == 0.8], movies, K_SHRINK, pd.to_datetime(VAL_CUTOFF, unit="s").year)
val_pool = val_known.loc[val_known["tier"] == 3, "movieId"].to_numpy()

test_train = pd.read_parquet(PROCESSED_DIR / f"benchmark_train_{VERSION}.parquet")
test_hits = pd.read_parquet(PROCESSED_DIR / f"benchmark_hits_{VERSION}.parquet").groupby("userId")["movieId"].apply(set).to_dict()
test_mean = test_train.groupby("userId")["rating"].mean()
at_cutoff = pd.read_parquet(PROCESSED_DIR / f"benchmark_movies_at_cutoff_{VERSION}.parquet").set_index("movieId")
test_pool = at_cutoff.index[at_cutoff["tier"] == 3].to_numpy()
popularity_share = (at_cutoff["rating_count"] / at_cutoff["rating_count"].sum()).to_dict()
print(f"validation: {len(val_hits):,} users, pool {len(val_pool):,} | test: {len(test_hits):,} users, pool {len(test_pool):,}")

## 2. Tuning on the validation split

Each configuration is trained on all pre-2016 ratings and scored on the 3,000 validation users. Results are saved row by row to `artifacts/cf_tuning_v1.csv` as each finishes, so an interrupted run resumes without redoing finished configurations. ALS fits take one to a few minutes each.

| Method | Grid |
|---|---|
| item-to-item | neighbours K ∈ {50, 200} × table ∈ {watched, liked} |
| PureSVD | factors k ∈ {32, 64, 128, 256} × table ∈ {watched, liked, rating} |
| ALS | factors ∈ {64, 128} × regularisation ∈ {0.01, 0.1} × alpha ∈ {1, 10} × table ∈ {watched, liked} |

After the first pass, each method's best setting was on the edge of its grid, so the grid was widened beyond those edges: item-to-item K = 500, PureSVD k = 16, ALS factors 256 and alpha 40 (liked table, the winning version for every method).

In [ ]:
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import CosineRecommender


def train_scorer(method, params, ratings):
    """Train one CF model on `ratings`; return (user_scores function, column index of movies, extras)."""
    m, ui, mi = cf.build_matrix(ratings, params["table"])
    if method == "item_knn":
        model = CosineRecommender(K=params["K"])
        model.fit(m, show_progress=False)
        sim = model.similarity.tocsr()
        return (lambda u, _r: np.asarray((m[ui[u]] @ sim).todense()).ravel()), mi, {"model": model}
    if method == "pure_svd":
        v = cf.pure_svd(m, params["factors"], seed=SEED)
        return cf.svd_user_scores(m, ui, v), mi, {"item_factors": v}
    if method == "als":
        model = AlternatingLeastSquares(factors=params["factors"], regularization=params["reg"], alpha=params["alpha"],
                                        iterations=15, random_state=SEED)
        model.fit(m, show_progress=False)
        uf, itf = np.asarray(model.user_factors), np.asarray(model.item_factors)
        return cf.factor_user_scores(uf, itf, ui), mi, {"user_factors": uf, "item_factors": itf, "user_index": ui}
    raise ValueError(method)


GRID = ([("item_knn", {"K": K, "table": t}) for K in (50, 200) for t in ("watched", "liked")]
        + [("pure_svd", {"factors": k, "table": t}) for k in (32, 64, 128, 256) for t in ("watched", "liked", "rating")]
        + [("als", {"factors": f, "reg": r, "alpha": a, "table": t})
           for f in (64, 128) for r in (0.01, 0.1) for a in (1.0, 10.0) for t in ("watched", "liked")])
# widened after the first pass: each method's best was on the edge of its grid (larger ALS factors/alpha,
# fewer PureSVD factors, more item-to-item neighbours), so the next values beyond the edge are tried too
GRID += ([("item_knn", {"K": 500, "table": "liked"})]
         + [("pure_svd", {"factors": 16, "table": "liked"})]
         + [("als", {"factors": f, "reg": 0.1, "alpha": a, "table": "liked"}) for f, a in ((256, 10.0), (128, 40.0), (256, 40.0))])

TUNE_PATH = ARTIFACTS_DIR / f"cf_tuning_{VERSION}.csv"
done = pd.read_csv(TUNE_PATH) if TUNE_PATH.exists() else pd.DataFrame(columns=["method", "params"])
for method, params in GRID:
    key = json.dumps(params, sort_keys=True)
    if ((done["method"] == method) & (done["params"] == key)).any():
        continue
    start = time.time()
    scorer, mi, _ = train_scorer(method, params, pre2016)
    fit_s = time.time() - start
    res = ev.evaluate(cf.make_scorer_recommender(scorer, val_pool, mi), val_train, val_hits, pool_size=len(val_pool))
    row = {"method": method, "params": key, "recall@10": res["recall@10"], "recall@50": res["recall@50"],
           "ndcg@10": res["ndcg@10"], "fit_seconds": round(fit_s, 1), "ms_per_user": res["ms_per_user"]}
    done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
    done.to_csv(TUNE_PATH, index=False)
    print(f"{method:9s} {key:70s} recall@10 {row['recall@10']:.4f} recall@50 {row['recall@50']:.4f} fit {fit_s:.0f}s")

# validation reference points for the same split (no training needed)
val_popularity = val_known.set_index("movieId")["rating_count"]
for name, recommend in {"popular": rc.make_ranked_list(val_pool, val_popularity),
                        "vibe_match": None}.items():
    if recommend is None:
        vibes = pd.read_parquet(PROCESSED_DIR / "vibe_scores_v2.parquet").set_index("movieId").drop(columns="source").astype(float)
        recommend = rc.make_vibe_match(vibes, np.intersect1d(val_pool, vibes.index), val_mean)
    r = ev.evaluate(recommend, val_train, val_hits, pool_size=len(val_pool))
    print(f"validation reference {name:10s}: recall@10 {r['recall@10']:.4f}, recall@50 {r['recall@50']:.4f}")
write_metadata(TUNE_PATH, validation_users=len(val_hits), selection="recall@10, ties by recall@50", grid=len(GRID))
tuning = pd.read_csv(TUNE_PATH)
best = (tuning.sort_values(["recall@10", "recall@50"], ascending=False).groupby("method").head(1).set_index("method"))
print("\nbest per method on validation:\n", best[["params", "recall@10", "recall@50", "fit_seconds"]].round(4).to_string())

## 3. CF + vibe: tuning the blend weight on validation

score = w × (ALS score) + (1 − w) × (vibe_match score), each standardised within the user first so their scales match. Movies without vibe scores count as neutral in the vibe part.
- w = 1 is pure ALS; w = 0 is pure vibe.
- w is chosen on the validation split (w = 0, 0.1, …, 1, by recall@10, ties by recall@50), never on the test.
- A blend that picks w = 1 would mean vibe adds nothing on top of CF.

In [ ]:
vibes = pd.read_parquet(PROCESSED_DIR / "vibe_scores_v2.parquet").set_index("movieId").drop(columns="source").astype(float)
best_als = json.loads(best.loc["als", "params"])
val_als, val_mi, _ = train_scorer("als", best_als, pre2016)
val_vibe = cf.vibe_user_scores(vibes, val_mi, val_mean)

BLEND_PATH = ARTIFACTS_DIR / f"cf_vibe_blend_tuning_{VERSION}.csv"
if BLEND_PATH.exists():
    print(f"{BLEND_PATH} already exists, loading.")
    blend_tuning = pd.read_csv(BLEND_PATH)
else:
    rows = []
    for w in np.round(np.arange(0, 1.01, 0.1), 1):
        rec = cf.make_scorer_recommender(cf.blend_user_scores([val_als, val_vibe], [w, 1 - w]), val_pool, val_mi)
        r = ev.evaluate(rec, val_train, val_hits, pool_size=len(val_pool))
        rows.append({"w_als": w, "recall@10": r["recall@10"], "recall@50": r["recall@50"], "ndcg@10": r["ndcg@10"]})
        print(f"w_als {w:.1f}: recall@10 {r['recall@10']:.4f}, recall@50 {r['recall@50']:.4f}")
    blend_tuning = pd.DataFrame(rows)
    blend_tuning.to_csv(BLEND_PATH, index=False)
    write_metadata(BLEND_PATH, als=best_als, selection="recall@10, ties by recall@50", note="validation split (2016)")
W_ALS = float(blend_tuning.sort_values(["recall@10", "recall@50"], ascending=False)["w_als"].iloc[0])
print(blend_tuning.round(4).to_string(index=False))
print(f"chosen blend weight: w_als = {W_ALS} (vibe weight {1 - W_ALS:.1f})")

## 4. Testing zone: the tuned methods on the 2018 benchmark

Each tuned method is retrained on **all ratings before 2018-01-04** (including the test users' own pre-2018 ratings, as in real use) and scored once on the 5,000 test users, in the same tier-3 pool as `popular / tier 3`. Results go on the scoreboard, with paired comparisons.

In [ ]:
def score(recommend, pool_size):
    return ev.evaluate(recommend, test_train, test_hits, vibe_vectors=vibe_vectors, popularity_share=popularity_share,
                       pool_size=pool_size, return_per_user=True)


vibe_vectors = dict(zip(vibes.index, rc.standardise(vibes).to_numpy()))
test_results, test_users = {}, {}
settings_base = {"cutoff": "2018-01-04", "users": len(test_hits), "seed": SEED, "pool_size": len(test_pool)}

# references, recomputed so the paired comparisons use the same users
test_results["popular"], test_users["popular"] = score(rc.make_ranked_list(test_pool, at_cutoff["rating_count"]), len(test_pool))
vibe_pool = np.intersect1d(test_pool, vibes.index)
test_results["vibe_match"], test_users["vibe_match"] = score(rc.make_vibe_match(vibes, vibe_pool, test_mean), len(vibe_pool))

scorers = {}
for method in ("item_knn", "pure_svd", "als"):
    params = json.loads(best.loc[method, "params"])
    start = time.time()
    scorers[method], test_mi, extras = train_scorer(method, params, pre2018)
    fit_s = time.time() - start
    if method == "als":                                                    # save the final ALS model for later steps
        np.save(MODELS_DIR / f"als_item_factors_{VERSION}.npy", extras["item_factors"])
        np.save(MODELS_DIR / f"als_user_factors_{VERSION}.npy", extras["user_factors"])
        pd.Series(test_mi.index, name="movieId").to_frame().to_parquet(MODELS_DIR / f"als_movie_index_{VERSION}.parquet", index=False)
        pd.Series(extras["user_index"].index, name="userId").to_frame().to_parquet(MODELS_DIR / f"als_user_index_{VERSION}.parquet", index=False)
        write_metadata(MODELS_DIR / f"als_item_factors_{VERSION}.npy", params=params, trained_on="ratings before 2018-01-04",
                       fit_seconds=round(fit_s, 1), note="row order = als_movie_index_v1.parquet; user factors in als_user_factors_v1.npy")
    test_results[method], test_users[method] = score(cf.make_scorer_recommender(scorers[method], test_pool, test_mi), len(test_pool))
    ev.append_scoreboard(SCOREBOARD, "7 collaborative filtering", f"{method} / tier 3", test_results[method],
                         settings={**settings_base, "params": params}, note=f"tuned on the 2016 validation split; fit {fit_s:.0f}s")

blend = cf.blend_user_scores([scorers["als"], cf.vibe_user_scores(vibes, test_mi, test_mean)], [W_ALS, 1 - W_ALS])
test_results["als+vibe"], test_users["als+vibe"] = score(cf.make_scorer_recommender(blend, test_pool, test_mi), len(test_pool))
ev.append_scoreboard(SCOREBOARD, "7 collaborative filtering", "als + vibe blend / tier 3", test_results["als+vibe"],
                     settings={**settings_base, "w_als": W_ALS}, note="blend weight tuned on the 2016 validation split")

summary = pd.DataFrame(test_results).T[["recall@10", "recall@50", "ndcg@10", "coverage@10", "diversity@10", "novelty@10", "ms_per_user"]]
print(summary.round(4).to_string())
print()
for a, b in [("als", "popular"), ("als", "vibe_match"), ("als", "pure_svd"), ("pure_svd", "item_knn"), ("als+vibe", "als")]:
    parts = []
    for metric in ("recall@10", "recall@50", "ndcg@10"):
        d, lo, hi, rel = ev.paired_gain(test_users[a], test_users[b], metric, seed=SEED)
        parts.append(f"{metric} {rel:+.0%} [{lo:+.4f}, {hi:+.4f}]")
    print(f"{a} vs {b}: " + " | ".join(parts))

**Results: testing zone (5,000 users, 2018 cutoff, tier-3 pool)**

| Method | recall@10 | recall@50 | ndcg@10 | coverage@10 | diversity@10 | novelty@10 | ms/user |
|---|---|---|---|---|---|---|---|
| popular | 0.0229 | 0.0682 | 0.0696 | 0.9% | 0.55 | 8.8 | 0.03 |
| vibe_match | 0.0291 | 0.0913 | 0.0669 | 5.8% | 0.25 | 17.0 | 0.79 |
| item-to-item (K = 500, liked) | 0.0395 | 0.1111 | 0.1129 | 2.5% | 0.50 | 9.7 | 0.50 |
| PureSVD (32 factors, liked) | 0.0493 | 0.1411 | 0.1369 | 3.4% | 0.55 | 10.0 | 0.38 |
| ALS (128 factors, reg 0.1, alpha 10, liked) | 0.0503 | 0.1457 | 0.1329 | **7.8%** | **0.63** | 10.8 | 0.57 |
| **ALS + vibe blend (w_ALS = 0.2)** | **0.0540** | **0.1615** | **0.1440** | 6.7% | 0.45 | 10.9 | 1.42 |

Paired comparisons (95% intervals):
- **ALS vs popular:** recall@10 +120%, recall@50 +114%, ndcg@10 +91%.
- **ALS vs vibe_match:** +73%, +60%, +99%. Learning from other users' ratings is far stronger than vibe alone.
- **PureSVD vs item-to-item:** +25%, +27%, +21%. Factor models clearly beat the simplest CF.
- **ALS vs PureSVD: effectively a tie.** recall@10 +2% (interval includes 0), recall@50 +3%, ndcg@10 −3% (both just significant, in opposite directions). ALS spreads recommendations over 2.3× more of the catalog (7.8% vs 3.4%) with higher diversity. PureSVD fits in 2 s instead of 54 s and has one setting.
- **ALS + vibe vs ALS alone: the core VibeRec claim holds.** recall@10 +7% [+0.0018, +0.0056], recall@50 +11% [+0.0126, +0.0190], ndcg@10 +8% [+0.0083, +0.0137]. Vibe adds real signal on top of CF.
  - **Costs:** diversity drops (0.63 → 0.45) and coverage drops a little (7.8% → 6.7%); about 1.4 ms per user.
  - **Blend weight:** chosen on validation, w_ALS = 0.2. The weights apply to standardised scores, so it is not a "20% CF" share; validation recall rises sharply from pure vibe (w = 0) to any blend and peaks at 0.2.

**Other findings**
- The "liked" table (only ratings above the user's own mean) was best for every method.
- On the 2016 validation split vibe_match alone was below popular, though above it on the 2018 test, so vibe alone is not stable across periods. As a complement to CF it helped on both splits.

**Limitations**
- MovieLens users stand in for VibeRec users.
- CF needs ratings, so new users and new movies get nothing from it (cold start, Step 8 onboarding).
- The Tag Genome's 2019 tags slightly favour the Genome-based vibe scores.
- Only the blend weight was tuned for the combination; the ranker in Step 10 can learn richer combinations.